# MLflow Pipeline Experiment Tracking

This notebook demonstrates a complete ML pipeline for customer churn prediction, tracked with MLflow. We perform preprocessing, model training, and track multiple runs with different hyperparameters.

## 1. Imports and Dataset Setup

Generating synthetic customer churn data.

In [ ]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import mlflow
import mlflow.sklearn

# Set seed for reproducibility
np.random.seed(42)

# Generate synthetic churn data
n_samples = 1000
data = pd.DataFrame({
    'tenure': np.random.randint(1, 72, n_samples),
    'monthly_charges': np.random.uniform(20, 120, n_samples),
    'contract_type': np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples),
    'internet_service': np.random.choice(['DSL', 'Fiber optic', 'None'], n_samples),
    'churn': np.random.binomial(1, 0.2, n_samples)
})

X = data.drop('churn', axis=1)
y = data['churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['tenure', 'monthly_charges']
categorical_features = ['contract_type', 'internet_service']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])


## 2. MLflow Experiment Initialization

Setting up the experiment name.

In [ ]:

mlflow.set_experiment("Churn_Prediction_Pipeline")


## 3. Training and Tracking Multiple Runs

We'll run three experiments with different models and hyperparameters.

In [ ]:

def train_and_track(model, model_name, params):
    with mlflow.start_run(run_name=model_name):
        pipeline = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', model)
        ])
        
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]
        
        metrics = {
            "accuracy": accuracy_score(y_test, y_pred),
            "f1": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_prob)
        }
        
        mlflow.log_params(params)
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipeline, "model")
        
        print(f"Run {model_name} logged with metrics: {metrics}")
        return mlflow.active_run().info.run_id

# Run 1: Baseline Random Forest
run_1_id = train_and_track(RandomForestClassifier(n_estimators=50, random_state=42), 
                "RandomForest_Baseline", 
                {"n_estimators": 50, "model_type": "RandomForest"})

# Run 2: Tuned Random Forest
run_2_id = train_and_track(RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42), 
                "RandomForest_Tuned", 
                {"n_estimators": 100, "max_depth": 10, "model_type": "RandomForest"})

# Run 3: Gradient Boosting
run_3_id = train_and_track(GradientBoostingClassifier(learning_rate=0.1, n_estimators=100, random_state=42), 
                "GradientBoosting_Tuned", 
                {"learning_rate": 0.1, "n_estimators": 100, "model_type": "GradientBoosting"})


## 4. Model Registration

Registering the best model based on ROC-AUC.

In [ ]:

# Find the best run based on ROC-AUC
runs = mlflow.search_runs(experiment_ids=[mlflow.get_experiment_by_name("Churn_Prediction_Pipeline").experiment_id])
best_run = runs.loc[runs['metrics.roc_auc'].idxmax()]
best_run_id = best_run['run_id']

print(f"Registering the best model from run {best_run_id} with ROC-AUC: {best_run['metrics.roc_auc']:.4f}")

model_uri = f"runs:/{best_run_id}/model"
mlflow.register_model(model_uri, "ChurnPredictionBestModel")
